In [2]:
import sqlite3

conn = sqlite3.connect("billing.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    barcode TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    price REAL NOT NULL
)
""")

products = [
    ("890101", "Rice", 50.0),
    ("890102", "Oil", 120.0),
    ("890103", "Sugar", 40.0),
    ("890104", "Milk", 30.0),
]

cursor.executemany(
    "INSERT OR IGNORE INTO products VALUES (?, ?, ?)",
    products
)

conn.commit()
conn.close()

print("✅ Database initialized successfully")


✅ Database initialized successfully


In [3]:
import sqlite3

GST_RATE = 0.05
DB_NAME = "billing.db"

cart = []

def get_product(barcode):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT name, price FROM products WHERE barcode = ?",
        (barcode,)
    )
    row = cursor.fetchone()
    conn.close()

    if row:
        return {"name": row[0], "price": row[1]}
    return None

def add_item(barcode):
    product = get_product(barcode)

    if not product:
        print("❌ Product not found!")
        return

    for item in cart:
        if item["barcode"] == barcode:
            item["qty"] += 1
            item["total"] = item["qty"] * item["price"]
            print(f"✔ {product['name']} quantity updated")
            return

    cart.append({
        "barcode": barcode,
        "name": product["name"],
        "price": product["price"],
        "qty": 1,
        "total": product["price"]
    })
    print(f"✔ {product['name']} added")

def calculate_totals():
    subtotal = sum(item["total"] for item in cart)
    gst = subtotal * GST_RATE
    grand_total = subtotal + gst
    return subtotal, gst, grand_total

def print_bill():
    print("\n========== ABC STORE ==========")
    print("Item        Qty   Price   Total")
    print("--------------------------------")

    for item in cart:
        print(f"{item['name']:<10} {item['qty']:<5} {item['price']:<7} {item['total']:.2f}")

    print("--------------------------------")
    subtotal, gst, grand_total = calculate_totals()

    print(f"Subtotal:        {subtotal:.2f}")
    print(f"GST (5%):        {gst:.2f}")
    print(f"Grand Total:     {grand_total:.2f}")
    print("================================")
    print("Thank You! Visit Again 🙏\n")

def main():
    print("🧾 BILLING SYSTEM (CLI + SQLITE)")
    print("Scan barcode | 'print' | 'exit'\n")

    while True:
        barcode = input("Scan Barcode: ").strip()

        if barcode.lower() == "exit":
            print("System closed")
            break

        elif barcode.lower() == "print":
            if not cart:
                print("⚠ Cart empty!")
            else:
                print_bill()
                cart.clear()
                print("🆕 New bill started\n")

        else:
            add_item(barcode)

if __name__ == "__main__":
    main()


🧾 BILLING SYSTEM (CLI + SQLITE)
Scan barcode | 'print' | 'exit'

✔ Rice added
✔ Rice quantity updated
✔ Milk added
✔ Sugar added

========== ABC STORE ==========
Item        Qty   Price   Total
--------------------------------
Rice       2     50.0    100.00
Milk       1     30.0    30.00
Sugar      1     40.0    40.00
--------------------------------
Subtotal:        170.00
GST (5%):        8.50
Grand Total:     178.50
Thank You! Visit Again 🙏

🆕 New bill started

System closed
